# Aurora — professeur : entraîneur (Colab)

Cette machine **entraîne** un vrai modèle de l'application sur ses points faibles mesurés, et ne décide jamais s'il est meilleur : ce verdict appartient à l'examinateur Kaggle.

Ce qu'elle reçoit : `diagnosis.json` (scores agrégés par compétence et empreintes des questions d'examen, jamais les questions). Ce qu'elle produit : un adaptateur LoRA, son reçu d'entraînement et le plan du curriculum.

Déroulement : *Exécution → Tout exécuter*. Choisir un runtime **GPU**. La cellule de contrôle refuse de démarrer si la mémoire mesurée ne suffit pas pour l'élève diagnostiqué : aucun GPU ni capacité n'est supposé.

Secrets Colab facultatifs (icône clé) : `HF_TOKEN` pour un modèle à accès restreint ; `KAGGLE_API_TOKEN` (jeton récent `KGAT_…`) ou bien `KAGGLE_USERNAME` et `KAGGLE_KEY` (kaggle.json) pour déposer l'adaptateur dans un jeu de données privé et lancer l'examen Kaggle automatiquement. Sans eux, l'adaptateur est seulement enregistré dans Google Drive (`Aurora_Professeur/`) et l'examen reste à lancer.

Cycle suivant : avec `CONTINUE_FROM = "compte/jeu-de-données"`, la machine télécharge l'adaptateur du cycle précédent et le diagnostic de ce candidat publiés par l'examinateur, puis **prolonge** cet adaptateur sur des exercices inédits au lieu de repartir du modèle de base. L'examinateur compare alors le nouvel adaptateur à la base et à l'adaptateur précédent sur les mêmes questions.

Élèves connus :
- `qwen2.5:7b` → `Qwen/Qwen2.5-7B-Instruct` (business, media, conversation)
- `qwen2.5-coder:7b` → `Qwen/Qwen2.5-Coder-7B-Instruct` (code, missions)
- `qwen3-coder:30b` → `Qwen/Qwen3-Coder-30B-A3B-Instruct` (code, missions, cowork)
- `qwen3-coder-next:q4_K_M` → `Qwen/Qwen3-Coder-Next` (code, missions)
- `orcarouter/Qwen3.8-27B-Uncensored` → `orcarouter/Qwen3.8-27B-Uncensored` (conversation, academy, cyber, cowork, business, media, engineering)
- `qwen3-vl:8b` → `Qwen/Qwen3-VL-8B-Instruct` (missions)
- `qwen3-vl:30b` → `Qwen/Qwen3-VL-30B-A3B-Instruct` (media)
- `deepseek-r1:32b` → `deepseek-ai/DeepSeek-R1-Distill-Qwen-32B` (cyber, academy, code)


In [ ]:
PROFILE = "standard"         # "standard" ou "full" ("smoke" = répétition technique seulement)
MODULES = None               # None = ceux du diagnostic ; sinon un sous-ensemble, par exemple "business,media"
CYCLE = 0
BUDGET_MINUTES = 300         # budget total ; l'adaptateur est sauvegardé avant la fin
OVERRIDES = {}               # par exemple {"curriculum": 8000, "epochs": 2}
CONTINUE_FROM = None         # jeu de données Kaggle privé "compte/nom" : reprendre l'adaptateur du cycle précédent


In [ ]:
import base64, json, os
from pathlib import Path
DIAGNOSIS_B64 = None
inputs = Path("/content/aurora_inputs")
inputs.mkdir(parents=True, exist_ok=True)
target = inputs / "diagnosis.json"
if DIAGNOSIS_B64:
    target.write_bytes(base64.b64decode(DIAGNOSIS_B64))
elif not target.is_file() and not CONTINUE_FROM:
    from google.colab import files
    print("Déposer le fichier diagnosis.json produit par l'examinateur Kaggle.")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise SystemExit("Un seul fichier diagnosis.json est attendu")
    target.write_bytes(next(iter(uploaded.values())))
try:
    from google.colab import userdata
    for name in ("HF_TOKEN", "KAGGLE_API_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY"):
        try:
            os.environ[name] = userdata.get(name)
        except Exception:
            pass
except ImportError:
    pass
if CONTINUE_FROM:
    # Cycle suivant : l'examinateur a publié l'adaptateur précédent et le diagnostic de ce candidat.
    if not (os.environ.get("KAGGLE_API_TOKEN") or (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"))):
        raise SystemExit("CONTINUE_FROM demande les secrets Kaggle pour télécharger l'adaptateur précédent ; sinon mettre CONTINUE_FROM = None")
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "kaggle", "six"], check=True)
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi()
    api.authenticate()
    continued = inputs / "continued"
    api.dataset_download_files(CONTINUE_FROM, path=str(continued), unzip=True, quiet=True)
    found = sorted(continued.glob("diagnosis_next_*.json"))
    if len(found) != 1 or not (continued / "adapter_config.json").is_file():
        raise SystemExit("Le jeu de données ne contient pas un adaptateur et son diagnostic diagnosis_next_N.json")
    for stale in continued.glob("init_adapter_*"):          # l'adaptateur d'avant ne sert pas à l'entraîneur
        stale.unlink()
    target = found[0]
diagnosis = json.loads(target.read_text())
if diagnosis.get("kind") != "aurora_professor_diagnosis" or not diagnosis.get("complete"):
    raise SystemExit("Diagnostic absent ou incomplet : relancer l'examinateur avant d'entraîner")
STUDENT = diagnosis["student"]["tag"]
print("Élève diagnostiqué :", STUDENT, "→", diagnosis["student"]["hf"],
      "| candidat déjà entraîné, reprise de son adaptateur" if CONTINUE_FROM else "| modèle de base")
for row in diagnosis["weak_points"][:12]:
    print(f"  {row['accuracy']:.3f}  n={row['items']:4d}  {row['skill']}")
AURORA_OPTIONS = {"role": "train", "student": STUDENT, "profile": PROFILE, "cycle": CYCLE, "continue": bool(CONTINUE_FROM),
                  "modules": MODULES or ",".join(diagnosis["modules"]), "budget_minutes": BUDGET_MINUTES, "overrides": OVERRIDES}


In [ ]:
import subprocess
query = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.free,memory.total", "--format=csv,noheader,nounits"],
                       capture_output=True, text=True)
gpus = [line.split(", ") for line in query.stdout.strip().splitlines()] if query.returncode == 0 else []
free = [float(g[1]) / 1024 for g in gpus]
print("GPU mesurés :", [(g[0], f"{float(g[1]) / 1024:.1f} Gio libres / {float(g[2]) / 1024:.1f}") for g in gpus] or "aucun")
need = diagnosis["student"]["four_bit_gb"] + 5.0
usable = sum(max(0.0, f - 1.5) for f in free)
if usable < need and diagnosis["student"]["params_b"] > 2.0:
    raise SystemExit(f"Mémoire GPU insuffisante pour {STUDENT} : {usable:.1f} Gio utilisables, {need:.1f} nécessaires en 4 bits. "
                     "Choisir un runtime plus grand, ou demander à l'examinateur un diagnostic pour un élève plus petit.")
print("Admission acceptée pour", STUDENT)


In [ ]:
# Aurora professor - generated, private. Sources are embedded and hash-checked; no credential is stored here.
import base64, hashlib, io, json, os, re, shutil, stat, subprocess, sys, time, zipfile
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path, PurePosixPath

def extract_bundle(raw, destination, expected_sha256, max_bytes=100 * 1024 * 1024):
    """Validate all paths and hashes before writing; preserve any different existing file."""
    if len(raw) > max_bytes or hashlib.sha256(raw).hexdigest() != expected_sha256:
        raise ValueError("Source bundle hash or size mismatch")
    destination = Path(destination).resolve()
    with zipfile.ZipFile(io.BytesIO(raw)) as archive:
        names = archive.namelist()
        if len(set(names)) != len(names) or sum(i.file_size for i in archive.infolist()) > max_bytes:
            raise ValueError("Invalid or oversized source bundle")
        manifest = json.loads(archive.read("source_manifest.json"))
        if not isinstance(manifest, dict) or set(names) != set(manifest) | {"source_manifest.json"}:
            raise ValueError("Source manifest does not match archive")
        pending = []
        for info in archive.infolist():
            relative = PurePosixPath(info.filename)
            if (str(relative) != info.filename or relative.is_absolute() or ".." in relative.parts or "\\" in info.filename
                    or stat.S_ISLNK(info.external_attr >> 16)
                    or (info.filename != "source_manifest.json" and relative.suffix != ".py")):
                raise ValueError("Unsafe source bundle member")
            target = destination.joinpath(*relative.parts)
            if not target.resolve().is_relative_to(destination):
                raise ValueError("Source bundle escapes destination")
            content = archive.read(info)
            if info.filename in manifest and hashlib.sha256(content).hexdigest() != manifest[info.filename]:
                raise ValueError("Source file hash mismatch")
            if target.exists() and target.read_bytes() != content:
                raise ValueError("Existing different source preserved")
            pending.append((target, content))
        for target, content in pending:
            target.parent.mkdir(parents=True, exist_ok=True)
            if not target.exists():
                with target.open("xb") as stream:
                    stream.write(content)
    return destination


OPTIONS = dict(AURORA_OPTIONS)
SOURCE_SHA256 = 'c8cab0354c93912b038cbbd8570e275a8fe5d9a4e83de38a593ef9fbb8f15dcd'
SOURCES = base64.b64decode('...')
BASE = Path('/content')
WORK = BASE / "aurora_professor"
WORK.mkdir(parents=True, exist_ok=True)
# Sources live outside the output folder: only evidence is published as kernel output.
REPO = extract_bundle(SOURCES, Path('/content/aurora_sources') / SOURCE_SHA256[:16], SOURCE_SHA256)
print("AURORA_BOOT " + json.dumps({"sources": SOURCE_SHA256, "role": OPTIONS["role"]}), flush=True)


def numbers(text):
    return tuple(int(part) for part in re.findall(r"\d+", text)[:3])


missing = []
for name, minimum in {'transformers': '4.57.0', 'peft': '0.13.0', 'bitsandbytes': '0.43.0', 'accelerate': '0.34.0', 'datasets': '3.0.0'}.items():
    try:
        current = version(name)
    except PackageNotFoundError:
        current = None
    if current is None or numbers(current) < numbers(minimum):
        missing.append(name + ">=" + minimum)
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True, timeout=900)

env = dict(os.environ, PYTHONPATH=str(REPO), AURORA_PROFESSOR_ALLOW_CODE_EXECUTION="1", HF_HOME='/content/hf_cache',
           TOKENIZERS_PARALLELISM="false", HF_HUB_DISABLE_PROGRESS_BARS="1", PYTHONUNBUFFERED="1")
try:                                        # optional, only for gated base models
    from kaggle_secrets import UserSecretsClient
    env["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

argv = [sys.executable, "-m", "auto_rl.professor.run", OPTIONS["role"], "--student", OPTIONS["student"],
        "--profile", OPTIONS["profile"], "--cycle", str(OPTIONS["cycle"]), "--work", str(WORK),
        "--budget-minutes", str(OPTIONS["budget_minutes"])]
if OPTIONS.get("modules"):
    argv += ["--modules", OPTIONS["modules"]]
if OPTIONS.get("skills"):
    argv += ["--skills", OPTIONS["skills"]]
for key, value in (OPTIONS.get("overrides") or {}).items():
    argv += ["--" + key.replace("_", "-"), str(value)]
inputs = Path('/content/aurora_inputs')
# An adapter kept under ``init_adapter`` is the one a continued training started from, never the candidate.
adapters = sorted(inputs.rglob("adapter_config.json")) if inputs.is_dir() else []
candidates = [path for path in adapters if path.parent.name != "init_adapter"]
earlier = [path for path in adapters if path.parent.name == "init_adapter"]
if OPTIONS["role"] == "train":
    wanted = "diagnosis_next_*.json" if OPTIONS.get("continue") else "diagnosis.json"
    found = sorted(inputs.rglob(wanted)) if inputs.is_dir() else []
    if len(found) != 1:
        raise SystemExit("Expected exactly one %s from the examiner, found %d" % (wanted, len(found)))
    argv += ["--diagnosis", str(found[0])]
    if OPTIONS.get("continue"):
        if len(candidates) != 1:
            raise SystemExit("Expected exactly one adapter to continue, found %d" % len(candidates))
        argv += ["--init-adapter", str(candidates[0].parent)]
if OPTIONS["role"] == "verify":
    if len(candidates) != 1:
        raise SystemExit("Expected exactly one adapter from the trainer, found %d" % len(candidates))
    adapter = WORK / "training" / "adapter"
    if adapter.exists():
        shutil.rmtree(adapter)
    shutil.copytree(candidates[0].parent, adapter)
    for receipt in (candidates[0].parent.parent / "train_receipt.json", candidates[0].parent / "train_receipt.json"):
        if receipt.is_file():                # kernel output layout, then flat dataset layout
            shutil.copy2(receipt, adapter.parent / "train_receipt.json")
            break
    argv += ["--adapter", str(adapter)]
    flat = sorted(inputs.rglob("init_adapter_config.json")) if inputs.is_dir() else []
    before = WORK / "training" / "init_adapter"
    if before.exists():
        shutil.rmtree(before)
    if len(earlier) == 1:                    # same unseen items for the adapter this candidate continues
        shutil.copytree(earlier[0].parent, before)
    elif len(flat) == 1:                     # flat dataset layout: files prefixed with ``init_``
        before.mkdir(parents=True)
        for path in sorted(flat[0].parent.glob("init_adapter_*")):
            shutil.copy2(path, before / path.name[len("init_"):])
    if before.is_dir():
        argv += ["--previous-adapter", str(before)]

started = time.time()
# Lines are relayed one by one so a notebook cell shows progress, not only a script log.
process = subprocess.Popen(argv, env=env, cwd=str(REPO), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors="replace")
beat = time.time()
with (WORK / "run.log").open("a", encoding="utf-8") as log:
    for line in process.stdout:
        log.write(line)
        log.flush()
        if line.startswith("AURORA_") or "Error" in line or "Traceback" in line or line.startswith("  File "):
            print(line, end="", flush=True)
        elif time.time() - beat > 120:           # a quiet phase (downloads, model load) stays observable
            beat = time.time()
            print("AURORA_HEARTBEAT " + json.dumps({"elapsed_seconds": round(beat - started), "last": line.strip()[-160:]}), flush=True)
code = process.wait()
if code:
    print("".join((WORK / "run.log").read_text(encoding="utf-8", errors="replace").splitlines(True)[-60:]), flush=True)
files = {}
for path in sorted(WORK.rglob("*")):
    if path.is_file():
        sha = hashlib.sha256()
        with path.open("rb") as stream:
            for block in iter(lambda: stream.read(1 << 22), b""):
                sha.update(block)
        files[path.relative_to(WORK).as_posix()] = sha.hexdigest()
disk = {p: shutil.disk_usage(p).free // 2 ** 30 for p in ("/tmp", str(BASE)) if os.path.isdir(p)}
manifest = {"kind": "aurora_professor_remote_run", "options": OPTIONS, "sources_sha256": SOURCE_SHA256, "exit_code": code,
            "elapsed_seconds": round(time.time() - started, 1), "free_disk_gb": disk, "files": files}
(BASE / "aurora_professor_manifest.json").write_text(json.dumps(manifest, indent=2))
print("AURORA_DONE " + json.dumps({k: manifest[k] for k in ("exit_code", "elapsed_seconds", "free_disk_gb")}), flush=True)
if code:
    raise SystemExit(code)


In [ ]:
import json, os, shutil, sys
from pathlib import Path
work = Path("/content/aurora_professor")
receipt = json.loads((work / "training" / "train_receipt.json").read_text())
print("Pas effectués :", receipt["steps"], "| arrêt :", receipt["stopped"], "| loss de développement :",
      round(receipt["initial_dev_loss"], 4), "→", round(receipt["selected_dev_loss"], 4))
print("Une loss plus basse ne prouve pas de meilleures réponses : seul l'examen apparié de Kaggle décide.")
archive = shutil.make_archive(f"/content/aurora_adaptateur_cycle{CYCLE}", "zip", root_dir=work, base_dir="training")
shutil.copy2(work / "curriculum.json", "/content/curriculum.json")
if os.environ.get("KAGGLE_API_TOKEN") or (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")):
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "kaggle", "six"], check=True)
    sys.path.insert(0, str(REPO))
    from auto_rl.professor import kaggle_runner
    owner = kaggle_runner.whoami()
    dataset = kaggle_runner.publish_adapter(owner, work / "training", student=STUDENT, note=f"cycle {CYCLE} {STUDENT}")
    print("Adaptateur déposé dans le jeu de données privé", dataset)
    folder = Path("/content/aurora_examinateur")
    kaggle_runner.write_kernel(folder, owner, "verify", student=STUDENT, profile=PROFILE, modules=AURORA_OPTIONS["modules"],
                               cycle=CYCLE, budget_minutes=240, dataset_sources=[dataset])
    print("Examen lancé :", kaggle_runner.launch(folder, timeout_seconds=240 * 60 + 1500))
else:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        target = Path("/content/drive/MyDrive/Aurora_Professeur")
        target.mkdir(exist_ok=True)
        shutil.copy2(archive, target / Path(archive).name)
        shutil.copy2("/content/curriculum.json", target / f"curriculum_cycle{CYCLE}.json")
        print("Adaptateur enregistré dans Google Drive :", target / Path(archive).name)
    except Exception as exc:
        print("Drive indisponible (", exc, ") : télécharger", archive, "depuis le panneau Fichiers.")
